# -----Importing the libraries ----#

In [ ]:
# ==============================================================
# ✅ USER CONFIG (edit only here)
# ============================================================== 

# --- Strategy name (used in folders / metadata) ---
strategy_name = "Bulls_strategy"

# --- Export root ---
# If your original notebook sets initial_export_path earlier, you can override it here.
# Example: initial_export_path = "<local-path>"
# This notebook writes to: initial_export_path + "/Trade Level Data/"
initial_export_path = initial_export_path

# --- Parallelization ---
cpu_count = 2  # start small; increase carefully (RAM will be the limiter)

# --- Universe controls ---
symbols_not_to_BT = ["XTI/USD","XNG/USD","USD/TRY","EUR/TRY","USD/CNH","TRY/JPY"]

# --- Timeframe / condition ---
tf = ["D1"]
condition = ["Long"]

# --- Parameter grid ---
parameter_1 = list(range(180, 220, 10))  # n Moving average
parameter_2 = list(range(4, 11, 2))      # ROC & RSI of ROC
parameter_3 = list(range(15, 30, 5))     # RSI(ROC) entry threshold
parameter_4 = list(range(50, 100, 10))   # RSI(ROC) exit threshold
parameter_5 = [10]                       # kept constant
parameter_6 = [10]                       # kept constant

# --- Optional batching (memory control) ---
# None  -> all symbols in one go (fastest)
# 200   -> process symbols in batches of 200 per parameter set (safer RAM)
symbol_batch_size = None

# ==============================================================


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import numpy as np
from matplotlib import style
import datetime as dt
from functools import reduce 
import time
import pandas as pd
from IPython.display import clear_output
import re
import ta
from scipy.signal import argrelextrema
from scipy.signal import find_peaks, peak_prominences
import seaborn as sns
%matplotlib inline
from sklearn import preprocessing
from scipy.spatial.distance import euclidean
# from fastdtw import fastdtw
from ta import *
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta
import calendar
from dateutil import relativedelta
import warnings
warnings.filterwarnings("ignore")
import itertools


from datetime import date, timedelta as td

from functools import reduce
import inspect
from functools import partial
from multiprocessing import Pool
# import talib

# ------ Rules ----------- #
Bulls_strategy

1. The close must be above the 200-day average.
2. The 5-day RSI indicator of the 5-day Rate of Change indicator must be below 20.
3. If 1 and 2 are true, then buy the close.
4. We sell at the close when the 5-day RSI is above 50.

In [ ]:
strategy_name = 'Bulls_strategy' # ----------- Edit the Strategy_name ------------- #

In [ ]:
# ---------- Paths -----------#

#-------------- export path -----------#

initial_export_path = "<local-path>" + strategy_name + "/"


export_path_Open_Trades =  initial_export_path + "/Open_Trades/"
export_path_trade_level_path =  initial_export_path + "/Trade Level Data/"
export_path_BT =  initial_export_path + "/BT/"
export_path_FT =  initial_export_path + "/FT/"
path_for_error_files = initial_export_path + "/Error/"

try:
    os.mkdir(initial_export_path)
    os.mkdir(export_path_Open_Trades)
    os.mkdir(export_path_trade_level_path)
    os.mkdir(export_path_BT)
    os.mkdir(export_path_FT)
    os.mkdir(path_for_error_files)

except:
    print("Folder Already Created")

# #-------------- Entry data export path -----------#
# Entry_data_export_path = "<local-path>"

#-------------- broker data export path -----------#
broker_data_export_path = "<local-path>"

# -----Loading the data -----#

In [ ]:

# ---------- d1 Data -----------#
d1 = pd.read_parquet(
    broker_data_export_path + "CMC_DAILY_OHLC_close_m20_m25_m30_updated.parquet"
)  # ------ it is Fwd looking data ------- #

d1['date'] = pd.to_datetime(d1['date'], format='%Y-%m-%d')
d1.rename(columns={'Symbol': 'Currency'}, inplace=True)
d1['Timeframe'] = "D1"

price_data = pd.concat([d1], axis=0, ignore_index=True)

price_data.tail()

In [ ]:
# optional (faster + less memory)
price_data["Timeframe"] = price_data["Timeframe"].astype("category")
price_data["Currency"]  = price_data["Currency"].astype("category")

# Build lookup ONCE:
PRICE_BY_KEY = {
    (tf, cur): g
    for (tf, cur), g in price_data.groupby(["Timeframe", "Currency"], sort=False)
}

strategy = [strategy_name]

# ✅ All symbols will be processed together inside Func_param()
symbols = price_data['Currency'].unique()
symbols = list(set(symbols) - set(symbols_not_to_BT))

# ✅ PARAM-LEVEL iteration list (Currency removed)
all_param_combinations = itertools.product(
    strategy, condition, tf,
    parameter_1, parameter_2, parameter_3,
    parameter_4, parameter_5, parameter_6
)

iter_list = pd.DataFrame(
    all_param_combinations,
    columns=[
        "Strategy","Condition","Timeframe",
        "parameter_1","parameter_2","parameter_3","parameter_4","parameter_5","parameter_6"
    ]
)

iter_list


In [ ]:
strategy = [strategy_name]


symbols = price_data['Currency'].unique()

symbols_not_to_BT = ["XTI/USD","XNG/USD","USD/TRY","EUR/TRY","USD/CNH","TRY/JPY"]

symbols = list(set(symbols) - set(symbols_not_to_BT))


condition = ["Long"]

tf = ["D1"]

# -------- Choose the parameter as per strategy -------------- #

parameter_1 = list(range(180,220,10)) # n Moving average
parameter_2 = list(range(4,11,2)) # ROC & RSI of ROC
parameter_3 = list(range(15,30,5)) # threshold for RSI of ROC.
parameter_4 = list(range(50,100,10)) # threshold for RSI of ROC exit
parameter_5 = list(range(10,11,1)) # parameter not required, keeping it constant.
parameter_6 = list(range(10,11,1)) # parameter not required, keeping it constant.


all_combinations = itertools.product(strategy,symbols, condition,tf,
                                     parameter_1,parameter_2,parameter_3,parameter_4,parameter_5,parameter_6)


iter_list = pd.DataFrame(all_combinations, columns=['Strategy','Currency','Condition','Timeframe',
                                                    'parameter_1','parameter_2','parameter_3','parameter_4',
                                                    'parameter_5','parameter_6'])



iter_list['Quote_Curr'] = iter_list['Currency'].str[4:]

# ----------- Reducing the iterations by Half ---------------- #
# Sort first
iter_list = iter_list.sort_values(by=["Strategy", "Currency"]).reset_index(drop=True)

# Add group-wise index
iter_list["_grp_idx"] = iter_list.groupby(["Strategy", "Currency"]).cumcount()

# Keep even iterations
iter_list = iter_list[iter_list["_grp_idx"] % 2 == 0].drop(columns="_grp_idx").reset_index(drop = True)
         
     

In [ ]:
completed_file_path = export_path_trade_level_path
error_completed_file_path = path_for_error_files

completed_file_1 = os.listdir(completed_file_path)
completed_file_2 = os.listdir(error_completed_file_path)
completed_file_2 = [f.replace("_error", "") for f in completed_file_2]

completed_set = set(completed_file_1) | set(completed_file_2)

# ----------------------------
# Build file_name vectorized + filter (PARAM-LEVEL)
# Naming:
#   ALL_<Timeframe>_<p1>_<p2>_<p3>_<p4>_<p5>_<p6>_<Condition>.parquet
# ----------------------------
file_name = (
    "ALL_" + iter_list["Timeframe"].astype(str) + "_" +
    iter_list["parameter_1"].astype(str) + "_" +
    iter_list["parameter_2"].astype(str) + "_" +
    iter_list["parameter_3"].astype(str) + "_" +
    iter_list["parameter_4"].astype(str) + "_" +
    iter_list["parameter_5"].astype(str) + "_" +
    iter_list["parameter_6"].astype(str) + "_" +
    iter_list["Condition"].astype(str) + ".parquet"
)

mask = ~file_name.isin(completed_set)
iteration_list = iter_list.loc[mask].reset_index(drop=True).values.tolist()

len(iteration_list)


In [ ]:
completed_file_path = export_path_trade_level_path
error_completed_file_path = path_for_error_files

completed_file_1 = os.listdir(completed_file_path)
completed_file_2 = os.listdir(error_completed_file_path)
completed_file_2 = [f.replace("_error", "") for f in completed_file_2]

completed_set = set(completed_file_1) | set(completed_file_2)

# print(len(completed_file_1), len(completed_file_2), len(completed_set))

# ----------------------------
# 2) Build file_name vectorized + filter
# ----------------------------
# IMPORTANT: iter_list is your DataFrame created from itertools.product
# Columns expected:
# ['Strategy','Currency','Condition','Timeframe','parameter_1','parameter_2','parameter_3','parameter_4','parameter_5','parameter_6']

cur = iter_list["Currency"].str.replace("/", "_", regex=False)
tf  = iter_list["Timeframe"].astype(str)
cond = iter_list["Condition"].astype(str)

p1 = iter_list["parameter_1"].astype(str)
p2 = iter_list["parameter_2"].astype(str)
p3 = iter_list["parameter_3"].astype(str)
p4 = iter_list["parameter_4"].astype(str)
p5 = iter_list["parameter_5"].astype(str)
p6 = iter_list["parameter_6"].astype(str)

iter_list["file_name"] = (
    cur + "_" + tf + "_" + p1 + "_" + p2 + "_" + p3 + "_" + p4 + "_" + p5 + "_" + p6 + "_" + cond + ".parquet"
)

# mask: True if already completed
done_mask = iter_list["file_name"].isin(completed_set)

# optional: quick visibility (don’t print millions)
print("Completed:", int(done_mask.sum()), "Pending:", int((~done_mask).sum()))

# ----------------------------
# 3) Final remaining iteration_list as list-of-lists
# ----------------------------
# Keep exact row structure you were using (all columns except file_name)
cols_out = ['Strategy','Currency','Condition','Timeframe',
            'parameter_1','parameter_2','parameter_3','parameter_4','parameter_5','parameter_6',
            'Quote_Curr']

# If Quote_Curr / SL Min Cap already exist in iter_list, this works directly.
# If not, add them as you did earlier before this block.

iteration_list = iter_list.loc[~done_mask, cols_out].to_numpy().tolist()

In [ ]:
# iteration_list = iteration_list[:1]
iteration_list

In [ ]:
len(iteration_list)

def Func_param(it, iteration_list, price_data, initial_export_path, symbols_universe=None, symbol_batch_size=None):
    # PARAM-LEVEL run:
    #   - Iterates ONLY on (Strategy, Condition, Timeframe, p1..p6)
    #   - Computes signals + trades for ALL symbols in one go (or in batches)
    #   - Exports ONE parquet with all instruments

    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    import pyarrow
    from IPython.display import clear_output

    i = iteration_list[it]
    Strategy    = i[0]
    condition   = i[1]
    timeframe   = i[2]
    parameter_1 = int(i[3])
    parameter_2 = int(i[4])
    parameter_3 = int(i[5])
    parameter_4 = int(i[6])
    parameter_5 = int(i[7])
    parameter_6 = int(i[8])

    # ----------------------------
    # Filter timeframe
    # ----------------------------
    df_all = price_data[price_data["Timeframe"] == timeframe].copy()
    df_all["date"] = pd.to_datetime(df_all["date"])
    df_all = df_all.sort_values(["Currency", "date"]).reset_index(drop=True)

    # Universe
    if symbols_universe is None:
        symbols_universe = df_all["Currency"].unique().tolist()
    else:
        symbols_universe = list(symbols_universe)

    # Optional batching
    if symbol_batch_size is None:
        symbol_batches = [symbols_universe]
    else:
        symbol_batches = [symbols_universe[i:i+symbol_batch_size] for i in range(0, len(symbols_universe), symbol_batch_size)]

    all_trades = []

    for _batch in symbol_batches:
        df = df_all[df_all["Currency"].isin(_batch)].copy()
        df = df.sort_values(["Currency", "date"]).reset_index(drop=True)

        # Attach params (same for all rows in this run)
        df["Strategy"]     = Strategy
        df["Condition"]    = condition
        df["parameter_1"]  = parameter_1
        df["parameter_2"]  = parameter_2
        df["parameter_3"]  = parameter_3
        df["parameter_4"]  = parameter_4
        df["parameter_5"]  = parameter_5
        df["parameter_6"]  = parameter_6

        g = df.groupby("Currency", sort=False)

        # ----------------------------
        # Vectorized n_MA using (w-1) past askclose + current close_m05
        # ----------------------------
        w = parameter_1
        n = parameter_2

        sum_past = g["askclose"].apply(lambda s: s.shift(1).rolling(window=w-1, min_periods=w-1).sum())
        sum_past = sum_past.reset_index(level=0, drop=True)

        df["n_MA"] = (sum_past + df["close_m05"]) / w

        # ----------------------------
        # Vectorized ROC: current close_m05 vs lag askclose[n]
        # ----------------------------
        lag = g["askclose"].shift(n)
        df["nD_ROC"] = (df["close_m05"] - lag) / lag * 100.0

        # ----------------------------
        # RSI of ROC (Wilder-style)
        # ----------------------------
        def rsi_wilder(x: pd.Series, window: int) -> pd.Series:
            delta = x.diff()
            gain = delta.clip(lower=0)
            loss = (-delta).clip(lower=0)
            avg_gain = gain.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
            avg_loss = loss.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
            rs = avg_gain / avg_loss
            return 100 - (100 / (1 + rs))

        df["nD_RSI_of_ROC"] = g["nD_ROC"].apply(lambda s: rsi_wilder(s, n)).reset_index(level=0, drop=True)

        # ----------------------------
        # Entry / Exit signals
        # ----------------------------
        df["Entry_signal"] = np.where(
            (df["close_m05"] > df["n_MA"]) & (df["nD_RSI_of_ROC"] < parameter_3),
            "Long",
            0
        )

        df["Trading Price Point"] = np.where(df["Entry_signal"] == "Long", df["askclose"], 0)
        df["TRADE_OPEN_Flag"] = np.where(df["Entry_signal"] == "Long", 1, 0)
        df["Exit_Signal"] = np.where(df["nD_RSI_of_ROC"] > parameter_4, 1, 0)

        # ----------------------------
        # For each ENTRY row: find first EXIT row after it (else use last row),
        # then apply your "if >3 bars then close at entry+3" rule
        # ----------------------------
        df["RSI_exit_status"] = np.nan

        exit_cols = [
            ("Exit_askclose", "askclose"),
            ("Exit_askopen", "askopen"),
            ("Exit_asklow", "asklow"),
            ("Exit_askhigh", "askhigh"),
            ("Exit_bidclose", "bidclose"),
            ("Exit_bidopen", "bidopen"),
            ("Exit_bidlow", "bidlow"),
            ("Exit_bidhigh", "bidhigh"),
            ("Closing Date", "date"),
        ]
        for out_col, _ in exit_cols:
            df[out_col] = np.nan

        def attach_exits(one: pd.DataFrame) -> pd.DataFrame:
            one = one.reset_index(drop=True)
            nrows = len(one)
            if nrows == 0:
                return one

            entry_idx = np.flatnonzero(one["TRADE_OPEN_Flag"].values == 1)
            if entry_idx.size == 0:
                return one

            exit_idx = np.flatnonzero(one["Exit_Signal"].values == 1)
            last_i = nrows - 1

            pos = np.searchsorted(exit_idx, entry_idx + 1, side="left")
            chosen = np.where(pos < exit_idx.size, exit_idx[pos], last_i)

            cap = entry_idx + 3
            chosen = np.where((chosen - entry_idx) > 3, np.minimum(cap, last_i), chosen)

            for (out_col, src_col) in exit_cols:
                one.loc[entry_idx, out_col] = one.loc[chosen, src_col].values

            one.loc[entry_idx, "RSI_exit_status"] = np.where(chosen == last_i, "RSI_exit_not_reached", "RSI_exit_reached")
            return one

        df = g.apply(attach_exits).reset_index(drop=True)

        # ----------------------------
        # Trade-level table
        # ----------------------------
        trade_df = df[df["RSI_exit_status"] == "RSI_exit_reached"].copy()
        if trade_df.empty:
            trade_df = df.head(0).copy()

        trade_df["Total Amount"] = 100000
        trade_df["Risk_Percent"] = 0.01
        trade_df["Units"] = (trade_df["Total Amount"] / trade_df["Trading Price Point"]).round()

        trade_df["Profit_Loss"] = np.where(
            trade_df["Entry_signal"] == "Long",
            (trade_df["Units"] * trade_df["Exit_bidclose"]) - (trade_df["Units"] * trade_df["Trading Price Point"]),
            (trade_df["Units"] * trade_df["Trading Price Point"]) - (trade_df["Units"] * trade_df["Exit_askclose"])
        )
        trade_df["Good_Bad_Flag"] = np.where(trade_df["Profit_Loss"] > 0, 1, 0)

        trade_df["date"] = pd.to_datetime(trade_df["date"])
        trade_df["Year"] = trade_df["date"].dt.year
        trade_df["EntryDate Year"] = trade_df["date"].dt.year
        trade_df["EntryDate Month"] = trade_df["date"].dt.month

        trade_df["Closing Date"] = pd.to_datetime(trade_df["Closing Date"])
        trade_df["ClosingDate Year"] = trade_df["Closing Date"].dt.year
        trade_df["ClosingDate Month"] = trade_df["Closing Date"].dt.month

        all_trades.append(trade_df)

    # ----------------------------
    # Combine batches + export ONE file for ALL symbols
    # ----------------------------
    if len(all_trades) > 0:
        trade_df = pd.concat(all_trades, ignore_index=True)
    else:
        trade_df = pd.DataFrame()

    export_path_trade_level_path = initial_export_path + "/Trade Level Data/"
    out_name = (
        "ALL_" + str(timeframe) + "_" +
        str(parameter_1) + "_" + str(parameter_2) + "_" + str(parameter_3) + "_" +
        str(parameter_4) + "_" + str(parameter_5) + "_" + str(parameter_6) + "_" +
        str(condition) + ".parquet"
    )

    pq.write_table(pyarrow.table(trade_df), export_path_trade_level_path + out_name, flavor='fastparquet')
    clear_output()


In [ ]:
cpu_count = cpu_count

def parallel_task(func, iteration_list, cpus, price_data, initial_export_path, symbols_universe=None, symbol_batch_size=None):
    import inspect
    with open(f'./tmp_func_Long.py', 'w') as file:
        file.write(inspect.getsource(func).replace(func.__name__, "task"))

    from tmp_func_Long import task

    if __name__ == '__main__':
        from multiprocessing import Pool
        from functools import partial

        func = partial(
            task,
            iteration_list=iteration_list,
            price_data=price_data,
            initial_export_path=initial_export_path,
            symbols_universe=symbols_universe,
            symbol_batch_size=symbol_batch_size
        )

        pool = Pool(processes=cpus)
        res = pool.map(func, range(len(iteration_list)))
        pool.close()
        pool.join()
        return res


In [ ]:

cpu_count = 2

def parallel_task(func, iteration_list,cpus):

    with open(f'./tmp_func_Long.py', 'w') as file:
        file.write(inspect.getsource(func).replace(func.__name__, "task"))
        
    from tmp_func_Long import task

    if __name__ == '__main__':
        func = partial(task)
        pool = Pool(processes=cpus)
        res = pool.starmap(func,([i,iteration_list,PRICE_BY_KEY,initial_export_path] for i in range(len(iteration_list))))
        pool.close()
        pool.join()
        return res
    else:
        raise "Not in Jupyter Notebook"
# iterable = [i for i in range(len(iteration_list))]
outputs = parallel_task(Func,iteration_list,cpus = cpu_count)